In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from src.config import MATHDIAL_TRAIN, MATHDIAL_VAL

train = pd.read_csv(MATHDIAL_TRAIN)
val = pd.read_csv(MATHDIAL_VAL) if MATHDIAL_VAL.exists() else None

print("Train shape:", train.shape)
if val is not None:
    print("Val shape:", val.shape)
print("\nColumns:", list(train.columns))
train.head(2)

Train shape: (2262, 11)

Columns: ['qid', 'scenario', 'question', 'ground_truth', 'student_incorrect_solution', 'student_profile', 'teacher_described_confusion', 'self-correctness', 'self-typical-confusion', 'self-typical-interactions', 'conversation']


,qid,scenario,question,ground_truth,student_incorrect_solution,student_profile,teacher_described_confusion,self-correctness,self-typical-confusion,self-typical-interactions,conversation
0,5000012,1,Nancy is filling an aquarium for her fish. She...,First calculate the volume of the aquarium by ...,The aquarium has a volume of 4 x 6 x 3 = 72 cu...,Steven is a 7th grade student. He has difficul...,He added a step after completing the problem.,Yes,3.0,3.0,"Teacher: (probing)Steven, If you had 4 of some..."
1,5000084,2,John is very unfit and decides to work up to d...,He needs to do 15*3=45 progressions\nThat will...,"To get to 15 reps, John will take 15 - 1 = 14 ...",Stephanie is a 7th grade student. She has diff...,She became fixated on a wrong calculation and ...,No,2.0,2.0,"Teacher: (probing)Stephanie, How many days wil..."


In [2]:
for col in train.columns:
    print(f"--- {col} ---")
    print(train[col].iloc[0] if not pd.isna(train[col].iloc[0]) else "<NaN>")
    print()

--- qid ---
5000012

--- scenario ---
1

--- question ---
Nancy is filling an aquarium for her fish. She fills it halfway and goes to answer the door. While she's gone, her cat knocks the aquarium over and spills half the water in it. Then Nancy comes back and triples the amount of water in the aquarium. If the aquarium is 4 feet long, 6 feet wide, and 3 feet high, how many cubic feet of water are in the aquarium?

--- ground_truth ---
First calculate the volume of the aquarium by multiplying its length, width and height: 4 ft * 6 ft * 3 ft = 72 cubic ft
Then figure out what proportion of the aquarium is full after the cat knocks it over: 1/2 * 1/2 = 1/4
Then figure out what proportion of the aquarium is full after Nancy refills it: 3 * 1/4 = 3/4
Now multiply the proportion of the aquarium that's full by the aquarium's volume to find out how much water is in it: 72 cubic ft * 3/4 = 54 cubic ft
 54

--- student_incorrect_solution ---
The aquarium has a volume of 4 x 6 x 3 = 72 cubic fee

In [3]:
# MathDial has a 'conversation' column with the full dialogue as text
sample = train.iloc[0]
print("Question:", sample.get("question", "?"))
print("\nGround truth:", sample.get("ground_truth", "?"))
print("\nStudent incorrect solution:", sample.get("student_incorrect_solution", "?"))
print("\nConversation:")
print(sample.get("conversation", "?")[:2000])

Question: Nancy is filling an aquarium for her fish. She fills it halfway and goes to answer the door. While she's gone, her cat knocks the aquarium over and spills half the water in it. Then Nancy comes back and triples the amount of water in the aquarium. If the aquarium is 4 feet long, 6 feet wide, and 3 feet high, how many cubic feet of water are in the aquarium?

Ground truth: First calculate the volume of the aquarium by multiplying its length, width and height: 4 ft * 6 ft * 3 ft = 72 cubic ft
Then figure out what proportion of the aquarium is full after the cat knocks it over: 1/2 * 1/2 = 1/4
Then figure out what proportion of the aquarium is full after Nancy refills it: 3 * 1/4 = 3/4
Now multiply the proportion of the aquarium that's full by the aquarium's volume to find out how much water is in it: 72 cubic ft * 3/4 = 54 cubic ft
 54

Student incorrect solution: The aquarium has a volume of 4 x 6 x 3 = 72 cubic feet.
When Nancy fills it halfway, she fills it with 72/2 = 36 cu

In [4]:
# Look for self-correctness and other quality fields
for col in ["self-correctness", "self_correctness", "self-typical-confusion",
            "self-typical-interactions", "scenario"]:
    if col in train.columns:
        print(f"{col} value counts:")
        print(train[col].value_counts(dropna=False).head(10))
        print()

self-correctness value counts:
self-correctness
Yes                                    1696
Yes, but I had to reveal the answer     303
No                                      254
NaN                                       9
Name: count, dtype: int64

self-typical-confusion value counts:
self-typical-confusion
4.0    657
3.0    584
5.0    490
2.0    364
1.0    158
NaN      9
Name: count, dtype: int64

self-typical-interactions value counts:
self-typical-interactions
4.0    666
3.0    613
5.0    503
2.0    307
1.0    164
NaN      9
Name: count, dtype: int64

scenario value counts:
scenario
1    469
2    456
3    451
4    445
5    441
Name: count, dtype: int64



In [5]:
def count_turns(conv_str):
    if not isinstance(conv_str, str):
        return 0, 0
    teacher = conv_str.count("Teacher:")
    student = conv_str.count("Student:")
    return teacher, student

train["n_teacher"] = train["conversation"].apply(lambda x: count_turns(x)[0])
train["n_student"] = train["conversation"].apply(lambda x: count_turns(x)[1])
print(train[["n_teacher", "n_student"]].describe())

         n_teacher    n_student
count  2262.000000  2262.000000
mean      6.591512     5.569850
std       2.977919     3.260235
min       3.000000     0.000000
25%       4.000000     3.000000
50%       6.000000     5.000000
75%       9.000000     8.000000
max      28.000000    27.000000
